In [25]:
import pandas as pd

In [26]:
data = pd.read_csv("clinical_analytics.csv")
print(data.shape)
data.head()

(81667, 13)


,Admit Source,Admit Type,Appt Start Time,Care Score,Check-In Time,Clinic Name,Department,Diagnosis Primary,Discharge Datetime new,Encounter Number,Encounter Status,Number of Records,Wait Time Min
0,Emergency Room,Emergency,2014-01-02 11:38:50 PM,2,2014-01-02 11:24:00 PM,Madison Center,Cardiology,NaN,NaN,P7P4KC587,Cancelled,1,14
1,Emergency Room,Emergency,2014-01-02 11:44:18 PM,2,2014-01-02 11:24:00 PM,Madison Center,Cardiology,NaN,NaN,P7P4KC587,Cancelled,1,20
2,Emergency Room,Emergency,2014-01-02 11:47:11 PM,2,2014-01-02 11:24:00 PM,Madison Center,Cardiology,NaN,NaN,P7P4KC587,Cancelled,1,23
3,Emergency Room,Emergency,2014-01-08 10:38:04 PM,4,2014-01-08 9:00:00 PM,Madison Center,Emergency,NaN,NaN,PK559C587,Cancelled,1,98
4,Emergency Room,Emergency,2014-01-09 12:00:26 AM,4,2014-01-08 10:47:00 PM,Madison Center,Emergency,NaN,NaN,48559C587,Cancelled,1,73


In [3]:
data["Admit Source"].unique()

array(['Emergency Room', 'Clinic Referral', 'Physician Referral', nan,
       'Outside Health Care Facility',
       'Admitted as transfer from another unit', 'Court/Law Enforcement',
       'Outside Hospital', 'Transfer from Critical Access Hospital',
       'Outside Home Health Agency', 'Information Unavailable',
       'Psych, Substance Abuse, or Rehab Hospital'], dtype=object)

In [4]:
data["Encounter Status"].unique()

array(['Cancelled', 'Admit', 'Preadmit', 'ED Discharged', 'Discharged',
       'Inpatient Discharged', 'OBS Discharged', 'Day Surg Discharged',
       nan], dtype=object)

In [5]:
data["Number of Records"].unique()

array([1])

In [6]:
data["Department"].unique()

array(['Cardiology', 'Emergency', 'Ophthalmology', 'Oral Surgery',
       'General Surgery', 'Neurosurgery', 'Urology', 'Otolaryngology',
       'Gastroenterology', 'Plastic Surgery', 'Neurology',
       'General Medicine', 'Rehabilitation', 'Nephrology',
       'Dental Surgery', 'Orthopedics', 'Cardiac Surgery'], dtype=object)

In [23]:
pip install dash-bootstrap-components

Note: you may need to restart the kernel to use updated packages.


In [29]:
# ================================
# 📦 IMPORTACIONES
# ================================
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc

# ================================
# 🎨 CONFIG GLOBAL
# ================================
px.defaults.template = "plotly_dark"

COLORS = {
    "primary": "#4F46E5",
    "secondary": "#06B6D4",
    "success": "#10B981",
    "warning": "#F59E0B",
    "danger": "#EF4444",
    "bg": "#0F172A",
    "card": "#1E293B",
    "text": "#E2E8F0"
}

# ================================
# 📥 CARGA DE DATOS (FIX FINAL)
# ================================
df = pd.read_csv("clinical_analytics.csv")
df.columns = df.columns.str.strip()

# 🔥 CONVERSIÓN SEGURA DE FECHAS (SIN WARNINGS Y SIN PERDER DATOS)
df["Appt Start Time"] = pd.to_datetime(
    df["Appt Start Time"],
    errors="coerce",
    dayfirst=False  # cambia a True si tu formato es DD/MM/YYYY
)

# eliminar solo valores realmente inválidos
df = df[df["Appt Start Time"].notna()]

# 🔍 DEBUG OPCIONAL
print("Filas cargadas:", len(df))
print(df["Appt Start Time"].head())

# ================================
# 🚀 APP
# ================================
app = Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])
server = app.server

# ================================
# 🎨 COMPONENTES
# ================================
def kpi_card(title, value, color):
    return dbc.Card(
        dbc.CardBody([
            html.H6(title, className="text-muted"),
            html.H2(value, style={"color": color, "fontWeight": "bold"})
        ]),
        style={
            "borderRadius": "15px",
            "background": COLORS["card"],
            "boxShadow": "0px 4px 15px rgba(0,0,0,0.4)"
        }
    )

def graph_card(graph):
    return dbc.Card(
        dbc.CardBody(graph),
        style={
            "borderRadius": "15px",
            "background": COLORS["card"],
            "boxShadow": "0px 4px 15px rgba(0,0,0,0.4)"
        }
    )

# ================================
# 🎨 LAYOUT
# ================================
app.layout = dbc.Container([

    dbc.Navbar(
        dbc.Container([
            html.Div([
                html.H3("🏥 Dashboard Clínico", className="fw-bold mb-0"),
                html.Small("Analítica avanzada de pacientes", className="text-muted")
            ])
        ]),
        color="dark",
        dark=True,
        className="mb-4"
    ),

    dbc.Card([
        dbc.CardBody([
            dbc.Row([

                dbc.Col([
                    html.Label("📅 Fechas"),
                    dcc.DatePickerRange(
                        id="filtro_fecha",
                        start_date=df["Appt Start Time"].min(),
                        end_date=df["Appt Start Time"].max(),
                    ),
                ], md=4),

                dbc.Col([
                    html.Label("🏥 Clínica"),
                    dcc.Dropdown(
                        id="filtro_clinica",
                        options=[{"label": c, "value": c} for c in df["Clinic Name"].dropna().unique()],
                        multi=True,
                    ),
                ], md=4),

                dbc.Col([
                    html.Label("🚑 Fuente"),
                    dcc.Dropdown(
                        id="filtro_fuente",
                        options=[{"label": s, "value": s} for s in df["Admit Source"].dropna().unique()],
                        multi=True,
                    ),
                ], md=4),

            ])
        ])
    ], className="mb-4", style={"background": COLORS["card"], "borderRadius": "15px"}),

    dbc.Row([
        dbc.Col(html.Div(id="kpi_pacientes"), md=4),
        dbc.Col(html.Div(id="kpi_espera"), md=4),
        dbc.Col(html.Div(id="kpi_satisfaccion"), md=4),
    ], className="mb-4"),

    dbc.Row([
        dbc.Col(graph_card(dcc.Graph(id="grafico_volumen")), md=6),
        dbc.Col(graph_card(dcc.Graph(id="grafico_tiempo")), md=6),
    ], className="mb-4"),

    dbc.Row([
        dbc.Col(graph_card(dcc.Graph(id="grafico_espera")), md=6),
        dbc.Col(graph_card(dcc.Graph(id="grafico_satisfaccion")), md=6),
    ])

], fluid=True)

# ================================
# 🔄 CALLBACK
# ================================
@app.callback(
    [
        Output("grafico_volumen", "figure"),
        Output("grafico_espera", "figure"),
        Output("grafico_satisfaccion", "figure"),
        Output("grafico_tiempo", "figure"),
        Output("kpi_pacientes", "children"),
        Output("kpi_espera", "children"),
        Output("kpi_satisfaccion", "children"),
    ],
    [
        Input("filtro_fecha", "start_date"),
        Input("filtro_fecha", "end_date"),
        Input("filtro_clinica", "value"),
        Input("filtro_fuente", "value"),
    ],
)
def actualizar_dashboard(inicio, fin, clinicas, fuentes):

    datos = df.copy()

    if inicio and fin:
        datos = datos[
            (datos["Appt Start Time"] >= inicio) &
            (datos["Appt Start Time"] <= fin)
        ]

    if clinicas:
        datos = datos[datos["Clinic Name"].isin(clinicas)]

    if fuentes:
        datos = datos[datos["Admit Source"].isin(fuentes)]

    # 📊 VOLUMEN
    volumen = datos.groupby("Clinic Name").size().reset_index(name="Pacientes")
    volumen = volumen.sort_values("Pacientes", ascending=False)

    fig_volumen = px.bar(
        volumen,
        x="Clinic Name",
        y="Pacientes",
        color="Pacientes",
        color_continuous_scale="Blues",
        title="Pacientes por Clínica"
    )

    # 📅 TENDENCIA
    tendencia = datos.groupby(datos["Appt Start Time"].dt.date).size().reset_index(name="Pacientes")

    fig_tiempo = px.line(
        tendencia,
        x="Appt Start Time",
        y="Pacientes",
        markers=True,
        title="Tendencia de Pacientes"
    )

    # ⏳ ESPERA
    fig_espera = px.box(
        datos,
        x="Department",
        y="Wait Time Min",
        color="Department",
        title="Tiempo de Espera"
    )

    # ⭐ SATISFACCIÓN
    fig_satisfaccion = px.histogram(
        datos,
        x="Care Score",
        nbins=20,
        color_discrete_sequence=[COLORS["secondary"]],
        title="Satisfacción del Paciente"
    )

    # KPIs
    total = len(datos)
    avg_wait = round(datos["Wait Time Min"].mean(), 2)
    avg_score = round(datos["Care Score"].mean(), 2)

    return (
        fig_volumen,
        fig_espera,
        fig_satisfaccion,
        fig_tiempo,
        kpi_card("Pacientes", total, COLORS["primary"]),
        kpi_card("Espera Promedio", avg_wait, COLORS["warning"]),
        kpi_card("Satisfacción", avg_score, COLORS["success"]),
    )

# ================================
# ▶️ RUN
# ================================
if __name__ == "__main__":
    app.run(debug=True, port=8051)

Filas cargadas: 81667
0   2014-01-02 23:38:50
1   2014-01-02 23:44:18
2   2014-01-02 23:47:11
3   2014-01-08 22:38:04
4   2014-01-09 00:00:26
Name: Appt Start Time, dtype: datetime64[ns]
